# Task 5 — Temporal Masked Autoencoder (TMAE)
**CPE342 · Account Security Monitoring · unsupervised anomaly detection**

แต่ละแถว = ผู้เล่น 1 คน มีข้อมูล 4 ช่วงเวลา (`_1`…`_4`) → มองเป็น **sequence 4 token + static token 1 ตัว**
แล้วฝึก Transformer ให้ "เดาช่วงที่ถูกปิด" จากช่วงอื่น ถ้าช่วง 3/4 ไม่สอดคล้องกับช่วงก่อนหน้า (EDA: anomaly ถูกใส่ในช่วง 3/4) → เดาผิดมาก → คะแนนผิดปกติสูง

```
period 1..4 : [27 ฟีเจอร์พฤติกรรม + lat, sin/cos lon + spatial (step, disp_home, invalid_lat)] → Linear → token
static      : [6 ฟีเจอร์ค่าเดียว]                                                         → Linear → token
            → + period embedding → Transformer Encoder (2 layers, d=64) → reconstruct
```

**ขั้นตอน:** โหลดข้อมูล → spatial time-series features → scale (scaler เดียวกันทั้ง 4 ช่วง) → train (masked reconstruction + iterative trimming) → score (ปิดช่วง 3 / 4 แล้ววัด error แบบ top-k) → ensemble กับ Isolation Forest → เขียนไฟล์

> ⚠️ Notebook นี้ **ไม่ submit อะไร** แค่เขียนไฟล์ลง `/kaggle/working/`
> รันบน CPU ได้ (~ไม่กี่นาที) — ไม่ต้องเปิด GPU

In [ ]:
# ===== Config =====
SEED        = 42
D_MODEL     = 64
N_HEADS     = 4
N_LAYERS    = 2
DROPOUT     = 0.1
EPOCHS      = 40
BATCH       = 512
LR          = 1e-3
WEIGHT_DECAY= 1e-4
P_PERIOD_MASK = 0.6     # โอกาสที่จะปิดทั้งช่วงเวลา 1 ช่วงต่อแถว
P_FEAT_MASK   = 0.15    # สัดส่วนช่องที่สุ่มปิดเพิ่ม (feature-level)
TRIM_FRAC     = 0.05    # ตัด 5% แถวที่ loss สูงสุดในแต่ละ batch (กันโมเดลจำ anomaly)
WARMUP_EPOCHS = 5       # ยังไม่ trim ช่วงแรก
CLIP_Z      = 20.0      # clip ค่าหลัง scale
TOPK        = 3         # คะแนน = ค่าเฉลี่ย error สูงสุด k ช่อง
SCORE_PERIODS = [3, 4]  # ช่วงที่ใช้ให้คะแนน (EDA: anomaly อยู่ช่วง 3/4)
W_STATIC    = 0.5       # น้ำหนัก error ของ static token
ENSEMBLE    = True      # rank-average กับ Isolation Forest บนฟีเจอร์การกระโดด
FLAG_RATES  = [0.035, 0.05, 0.08]   # สัดส่วนที่ flag เป็น anomaly (เลือกด้วย P5 บน LB)
OUT_DIR     = "/kaggle/working"

In [ ]:
import os, glob, random, time, warnings
import numpy as np, pandas as pd
warnings.filterwarnings("ignore", category=RuntimeWarning)   # nanmean ของแถวที่ว่างทั้งแถว

def find_file(name, must_contain=None):
    roots = ["/kaggle/input", "."]
    for r in roots:
        hits = glob.glob(os.path.join(r, "**", name), recursive=True)
        if must_contain:
            hits = [h for h in hits if any(m in h for m in must_contain)]
        if hits:
            return sorted(hits)[0]
    raise FileNotFoundError(f"หา {name} ไม่เจอ — Add Data: competition 'player-intelligence-system-cpe-342-ml'")

TEST_PATH   = find_file("test.csv", must_contain=["05_anomaly", "task5"])
SAMPLE_PATH = find_file("sample_submission.csv")
print(TEST_PATH, SAMPLE_PATH, sep="\n")

raw = pd.read_csv(TEST_PATH)
ids = pd.read_csv(SAMPLE_PATH)["id"]
assert len(raw) == len(ids), "จำนวนแถว test ไม่ตรง sample_submission"
# จับคู่ตามลำดับแถว: test แถวที่ i  ↔  ANS{i}  (ห้าม sort / drop แถว)
X = raw.drop(columns=["player_id"])
print(X.shape, "| missing rate ≈", round(X.isna().mean().mean(), 3))

## 1. Spatial time-series features + จัดรูปเป็น tensor `[N, 4 periods, C]`

In [ ]:
PERIODS = [1, 2, 3, 4]
bases = sorted({c.rsplit("_", 1)[0] for c in X.columns if c[-2:] in ("_1", "_2", "_3", "_4")})
STATIC = [c for c in X.columns if c[-2:] not in ("_1", "_2", "_3", "_4")]
NONGEO = [b for b in bases if b not in ("login_lat", "login_lon")]

def wrap180(x):
    return (x + 180.0) % 360.0 - 180.0

lat = np.stack([X[f"login_lat_{p}"].values for p in PERIODS], 1)   # [N,4]
lon = np.stack([X[f"login_lon_{p}"].values for p in PERIODS], 1)

def disp_deg(lat_a, lon_a, lat_b, lon_b):
    # ระยะแบบองศา (lon วนรอบ ±180) — ใช้ได้แม้ lat > 90 (anomaly ที่ไม่ clamp)
    return np.hypot(lat_b - lat_a, wrap180(lon_b - lon_a))

step = np.full_like(lat, np.nan)       # ระยะจากช่วงก่อนหน้า
disp_home = np.full_like(lat, np.nan)  # ระยะจาก "บ้าน" = ค่าเฉลี่ยช่วงก่อนหน้า
for i in range(1, 4):
    step[:, i] = disp_deg(lat[:, i-1], lon[:, i-1], lat[:, i], lon[:, i])
    with np.errstate(all="ignore"):
        h_lat = np.nanmean(lat[:, :i], 1)
        h_lon = np.degrees(np.arctan2(np.nanmean(np.sin(np.radians(lon[:, :i])), 1),
                                      np.nanmean(np.cos(np.radians(lon[:, :i])), 1)))
    disp_home[:, i] = disp_deg(h_lat, h_lon, lat[:, i], lon[:, i])
invalid_lat = np.where(np.isnan(lat), np.nan, (np.abs(lat) > 90).astype(float))

chans = {}
for b in NONGEO:
    chans[b] = np.stack([X[f"{b}_{p}"].values for p in PERIODS], 1)
chans["lat"] = lat
chans["lon_sin"] = np.sin(np.radians(lon))
chans["lon_cos"] = np.cos(np.radians(lon))
chans["log_step"] = np.log1p(step)
chans["log_disp_home"] = np.log1p(disp_home)
chans["invalid_lat"] = invalid_lat

CH = list(chans)
XP = np.stack([chans[c] for c in CH], -1).astype(np.float32)   # [N,4,C]
XS = X[STATIC].values.astype(np.float32)                        # [N,S]
print("period channels C =", len(CH), "| static S =", len(STATIC), "| XP", XP.shape)

# summary spatial features (ใช้ใน IF + sanity check)
geo_jump3 = disp_home[:, 2] > 5
geo_jump4 = disp_home[:, 3] > 5
print("geo jump p3:", int(np.nansum(geo_jump3)), "| p4:", int(np.nansum(geo_jump4)),
      "| invalid lat rows:", int(np.nan_to_num(invalid_lat).max(1).sum()))

In [ ]:
# ===== Scale: robust (median/IQR) ต่อ channel โดยใช้ scaler เดียวกันทั้ง 4 ช่วง =====
def robust_fit(a):
    a = a[~np.isnan(a)]
    med = np.median(a); q1, q3 = np.percentile(a, [25, 75]); iqr = q3 - q1
    if iqr < 1e-8:
        iqr = a.std() if a.std() > 1e-8 else 1.0
    return med, iqr

MP = (~np.isnan(XP)).astype(np.float32)        # 1 = มีค่า
MS = (~np.isnan(XS)).astype(np.float32)
XPs = np.zeros_like(XP); XSs = np.zeros_like(XS)
for c in range(XP.shape[2]):
    med, iqr = robust_fit(XP[:, :, c].ravel())
    XPs[:, :, c] = np.clip((XP[:, :, c] - med) / iqr, -CLIP_Z, CLIP_Z)
for c in range(XS.shape[1]):
    med, iqr = robust_fit(XS[:, c])
    XSs[:, c] = np.clip((XS[:, c] - med) / iqr, -CLIP_Z, CLIP_Z)
XPs = np.nan_to_num(XPs); XSs = np.nan_to_num(XSs)   # ค่าว่าง → 0 แต่ mask = 0 (โมเดลรู้ว่าว่าง)
print("observed ratio:", MP.mean().round(3), MS.mean().round(3))

## 2. Model

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
seed_all(SEED)
torch.set_num_threads(max(1, os.cpu_count() or 1))
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)

class TMAE(nn.Module):
    # token = Linear([value*mask, mask]) — ช่องที่ว่าง/ถูกปิดจึงไม่ต้องเติมค่า
    def __init__(self, C, S, d=64, heads=4, layers=2, dropout=0.1):
        super().__init__()
        self.per_in = nn.Linear(2 * C, d)
        self.st_in  = nn.Linear(2 * S, d)
        self.pos    = nn.Parameter(torch.randn(1, 5, d) * 0.02)   # 4 periods + static
        self.mask_tok = nn.Parameter(torch.randn(1, 1, d) * 0.02)
        layer = nn.TransformerEncoderLayer(d, heads, 4 * d, dropout, activation="gelu",
                                           batch_first=True, norm_first=True)
        self.enc  = nn.TransformerEncoder(layer, layers, enable_nested_tensor=False)
        self.norm = nn.LayerNorm(d)
        self.per_out = nn.Linear(d, C)
        self.st_out  = nn.Linear(d, S)

    def forward(self, xp, vp, xs, vs, hide_period):
        # xp [B,4,C], vp = visible mask [B,4,C], hide_period [B,4] bool
        tp = self.per_in(torch.cat([xp * vp, vp], -1))
        tp = torch.where(hide_period.unsqueeze(-1), self.mask_tok.expand_as(tp), tp)
        ts = self.st_in(torch.cat([xs * vs, vs], -1)).unsqueeze(1)
        h = self.norm(self.enc(torch.cat([tp, ts], 1) + self.pos))
        return self.per_out(h[:, :4]), self.st_out(h[:, 4])

C, S = XPs.shape[2], XSs.shape[1]
model = TMAE(C, S, D_MODEL, N_HEADS, N_LAYERS, DROPOUT).to(DEVICE)
print(f"params: {sum(p.numel() for p in model.parameters()):,}")

tXP = torch.tensor(XPs); tMP = torch.tensor(MP); tXS = torch.tensor(XSs); tMS = torch.tensor(MS)
N = len(tXP)

## 3. Train — masked reconstruction + iterative trimming

In [ ]:
def make_masks(mp, ms):
    B = mp.shape[0]
    hide = torch.zeros(B, 4, dtype=torch.bool, device=mp.device)
    rows = torch.rand(B, device=mp.device) < P_PERIOD_MASK
    hide[rows, torch.randint(0, 4, (int(rows.sum()),), device=mp.device)] = True
    drop_p = (torch.rand_like(mp) < P_FEAT_MASK) & mp.bool()
    drop_s = (torch.rand_like(ms) < P_FEAT_MASK) & ms.bool()
    vp = mp * (~drop_p) * (~hide).unsqueeze(-1)
    vs = ms * (~drop_s)
    tgt_p = mp * (drop_p | hide.unsqueeze(-1))       # ทายเฉพาะช่องที่มีค่าจริงแต่ถูกปิด
    tgt_s = ms * drop_s
    return hide, vp, vs, tgt_p, tgt_s

def per_sample_loss(rp, rs, xp, xs, tgt_p, tgt_s):
    lp = F.smooth_l1_loss(rp, xp, reduction="none") * tgt_p
    ls = F.smooth_l1_loss(rs, xs, reduction="none") * tgt_s
    num = lp.sum((1, 2)) + ls.sum(1)
    den = tgt_p.sum((1, 2)) + tgt_s.sum(1)
    return num / den.clamp(min=1)

opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, total_steps=EPOCHS * ((N + BATCH - 1) // BATCH))
hist = []
t0 = time.time()
for ep in range(EPOCHS):
    model.train(); perm = torch.randperm(N); tot = 0.0
    for i in range(0, N, BATCH):
        b = perm[i:i + BATCH]
        xp, mp, xs, ms = tXP[b].to(DEVICE), tMP[b].to(DEVICE), tXS[b].to(DEVICE), tMS[b].to(DEVICE)
        hide, vp, vs, tgt_p, tgt_s = make_masks(mp, ms)
        rp, rs = model(xp, vp, xs, vs, hide)
        l = per_sample_loss(rp, rs, xp, xs, tgt_p, tgt_s)
        if ep >= WARMUP_EPOCHS and TRIM_FRAC > 0:      # ตัดแถว loss สูงสุด (น่าจะเป็น anomaly)
            keep = l <= torch.quantile(l.detach(), 1 - TRIM_FRAC)
            l = l[keep]
        loss = l.mean()
        opt.zero_grad(); loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step(); sched.step(); tot += loss.item() * len(b)
    hist.append(tot / N)
    if ep % 5 == 0 or ep == EPOCHS - 1:
        print(f"epoch {ep+1:3d}/{EPOCHS}  loss {hist[-1]:.4f}  ({time.time()-t0:.0f}s)")

## 4. Score — ปิดทีละช่วง แล้ววัด error (normalize ต่อ channel, top-k)

In [ ]:
@torch.no_grad()
def recon_errors(hide_idx=None, hide_static=False, bs=4096):
    # คืน error |x - x̂| ของช่วง hide_idx [N,C] หรือของ static [N,S]; ช่องที่ไม่มีค่า = NaN
    model.eval(); out = []
    for i in range(0, N, bs):
        xp, mp, xs, ms = tXP[i:i+bs].to(DEVICE), tMP[i:i+bs].to(DEVICE), tXS[i:i+bs].to(DEVICE), tMS[i:i+bs].to(DEVICE)
        hide = torch.zeros(len(xp), 4, dtype=torch.bool, device=DEVICE)
        vp, vs = mp.clone(), ms.clone()
        if hide_idx is not None:
            hide[:, hide_idx] = True; vp[:, hide_idx] = 0
        if hide_static:
            vs = torch.zeros_like(ms)
        rp, rs = model(xp, vp, xs, vs, hide)
        if hide_static:
            e = (rs - xs).abs(); e[ms == 0] = float("nan")
        else:
            e = (rp[:, hide_idx] - xp[:, hide_idx]).abs(); e[mp[:, hide_idx] == 0] = float("nan")
        out.append(e.cpu().numpy())
    return np.concatenate(out)

def topk_score(err, k=TOPK):
    # robust z ต่อ channel → เฉลี่ย k ค่าสูงสุด (anomaly กระทบไม่กี่ channel)
    med = np.nanmedian(err, 0); mad = np.nanmedian(np.abs(err - med), 0) * 1.4826 + 1e-6
    z = np.clip((err - med) / mad, 0, None)
    z = np.where(np.isnan(z), -np.inf, z)
    top = -np.sort(-z, 1)[:, :k]
    top = np.where(np.isinf(top), 0, top)
    return top.mean(1), z

period_scores, period_z = {}, {}
for p in SCORE_PERIODS:
    period_scores[p], period_z[p] = topk_score(recon_errors(hide_idx=p - 1))
static_score, static_z = topk_score(recon_errors(hide_static=True), k=2)

tmae_score = np.max(np.stack([period_scores[p] for p in SCORE_PERIODS], 1), 1) + W_STATIC * static_score
print(pd.Series(tmae_score).describe().round(2))

## 5. Isolation Forest บนฟีเจอร์การกระโดด + Ensemble (rank average)

In [ ]:
from sklearn.ensemble import IsolationForest
from scipy.stats import rankdata

Fj = {}
for b in NONGEO:
    v = np.stack([X[f"{b}_{p}"].values for p in PERIODS], 1)
    for k in (3, 4):
        with np.errstate(all="ignore"):
            j = v[:, k-1] - np.nanmean(v[:, :k-1], 1)
        m = np.nanmedian(j); s = 1.4826 * np.nanmedian(np.abs(j - m)) + 1e-9
        Fj[f"{b}_jz{k}"] = np.abs((j - m) / s)
for k in (2, 3, 4):
    Fj[f"geo_disp{k}"] = np.log1p(disp_home[:, k-1])
Fj["invalid_lat"] = np.nanmax(np.nan_to_num(invalid_lat), 1)
for c in STATIC:
    Fj[c] = X[c].values
Zj = pd.DataFrame(Fj).fillna(0).clip(upper=50)
iforest = IsolationForest(n_estimators=500, max_samples=1024, random_state=SEED, n_jobs=-1).fit(Zj)
if_score = -iforest.score_samples(Zj)

rank = lambda s: rankdata(s) / len(s)
final_score = (rank(tmae_score) + rank(if_score)) / 2 if ENSEMBLE else tmae_score
print("Spearman(TMAE, IF) =", round(pd.Series(tmae_score).corr(pd.Series(if_score), method="spearman"), 3))

## 6. Sanity check (ไม่ใช่ ground truth!)
ใช้กลุ่ม "ผิดปกติชัดเจน" จาก EDA: ตำแหน่งกระโดด > 5° ในช่วง 3/4, lat > 90, หรือฟีเจอร์ช่วง 4 กระโดด (|z|>6) ≥ 2 ตัว
**ใช้ดูความสมเหตุสมผลเท่านั้น อย่าจูนตามนี้** — กลุ่มนี้เรานิยามเอง ตัวตัดสินจริงคือ P5 บน LB

In [ ]:
shock = np.stack([Fj[f"{b}_jz4"] > 6 for b in NONGEO], 1).sum(1)
proxy = (np.nan_to_num(disp_home[:, 2]) > 5) | (np.nan_to_num(disp_home[:, 3]) > 5) | (Fj["invalid_lat"] > 0) | (shock >= 2)
print(f"proxy-obvious rows: {proxy.sum()} ({proxy.mean():.1%})\n")
rows = []
for name, s in [("TMAE", tmae_score), ("IF-jump", if_score), ("Ensemble", final_score)]:
    r = {"model": name}
    for rate in FLAG_RATES:
        top = s >= np.quantile(s, 1 - rate)
        r[f"recall@{rate:.1%}"] = round((top & proxy).sum() / proxy.sum(), 3)
    rows.append(r)
pd.DataFrame(rows)

## 7. ตีความ — anomaly แต่ละแถวผิดปกติที่ช่วงไหน / channel ไหน (ใช้ใน report)

In [ ]:
def explain(i, n=3):
    items = []
    for p in SCORE_PERIODS:
        for c, z in enumerate(period_z[p][i]):
            if np.isfinite(z): items.append((z, f"p{p}:{CH[c]}"))
    for c, z in enumerate(static_z[i]):
        if np.isfinite(z): items.append((z * W_STATIC, f"static:{STATIC[c]}"))
    return ", ".join(f"{name}({z:.0f})" for z, name in sorted(items, reverse=True)[:n])

top_idx = np.argsort(-final_score)[:15]
pd.DataFrame({"id": ids.values[top_idx], "score": final_score[top_idx].round(3),
              "top reasons (robust z)": [explain(i) for i in top_idx]})

## 8. เขียนไฟล์ (ยังไม่ submit)

In [ ]:
os.makedirs(f"{OUT_DIR}/preds", exist_ok=True); os.makedirs(f"{OUT_DIR}/probes", exist_ok=True)
pd.DataFrame({"id": ids, "tmae": tmae_score, "iforest": if_score, "final": final_score}) \
  .to_csv(f"{OUT_DIR}/preds/task5_tmae_scores.csv", index=False)

# probe = mask task 1–4 (ค่าที่ให้คะแนน 0 / task3 = 0.5) + task5 จริง  → ใช้คู่กับ P0_all_mask.csv
MASK = {"task1": 2, "task2": 9, "task3": 0, "task4": 9}
tag = "tmae_ens" if ENSEMBLE else "tmae"
for rate in FLAG_RATES:
    pred = (final_score >= np.quantile(final_score, 1 - rate)).astype(int)
    pd.DataFrame({"id": ids, "pred": pred}).to_csv(f"{OUT_DIR}/preds/task5_{tag}_c{rate*1000:03.0f}.csv", index=False)
    probe = pd.DataFrame({"id": ids, **{k: v for k, v in MASK.items()}, "task5": pred})
    path = f"{OUT_DIR}/probes/P5_task5_only_mask_{tag}_c{rate*1000:03.0f}.csv"
    probe.to_csv(path, index=False)
    print(f"{path}  flag {pred.sum()} ({pred.mean():.1%})")
torch.save(model.state_dict(), f"{OUT_DIR}/tmae_task5.pt")
print("\nเสร็จ — ไม่ได้ submit อะไร ดาวน์โหลดไฟล์จาก Output แล้วส่งเองเมื่อพร้อม")

### Loss curve

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(6, 3)); plt.plot(range(1, len(hist) + 1), hist, lw=2)
plt.axvline(WARMUP_EPOCHS + 0.5, ls="--", c="gray", lw=1); plt.text(WARMUP_EPOCHS + 1, max(hist) * .95, "start trimming", fontsize=8, c="gray")
plt.xlabel("epoch"); plt.ylabel("masked recon loss"); plt.title("TMAE training"); plt.grid(alpha=.3); plt.show()